# Hansen Ch.10 Resampling — 计算框架

理论见 md。

下列 cell 演示 **jackknife / bootstrap SE**（Nerlove $\theta=\beta_3+\beta_4+\beta_5$）与 DDK cluster bootstrap 骨架。

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")
ner = pd.read_excel(ROOT / "Nerlove1963/Nerlove1963.xlsx")
for c in ner.columns:
    ner[c] = pd.to_numeric(ner[c], errors="coerce")
ner = ner.dropna().reset_index(drop=True)
y = np.log(ner.Cost.values)
X = np.column_stack([np.ones(len(ner)), np.log(ner.output), np.log(ner.Plabor),
                     np.log(ner.Pcapital), np.log(ner.Pfuel)])
n, k = X.shape
beta = np.linalg.lstsq(X, y, rcond=None)[0]
e = y - X @ beta
XXinv = np.linalg.inv(X.T @ X)
h = np.sum(X * (X @ XXinv), axis=1)
u = X * (e / np.clip(1 - h, 1e-12, None))[:, None]
V_as = (n / (n - k)) * XXinv @ (u.T @ u) @ XXinv
theta = beta[2] + beta[3] + beta[4]
R = np.array([0, 0, 1, 1, 1.0])
se_as = float(np.sqrt(R @ V_as @ R))
print("theta =", theta, "asym SE =", se_as)

# Jackknife
thetas = []
for i in range(n):
    Xi = np.delete(X, i, axis=0)
    yi = np.delete(y, i)
    bi = np.linalg.lstsq(Xi, yi, rcond=None)[0]
    thetas.append(bi[2] + bi[3] + bi[4])
thetas = np.array(thetas)
se_jack = np.sqrt((n - 1) / n * np.sum((thetas - thetas.mean()) ** 2))
print("jackknife SE =", se_jack)

# Bootstrap (pairs)
B = 400
rng = np.random.default_rng(0)
th_b = np.empty(B)
for b in range(B):
    idx = rng.integers(0, n, n)
    bb = np.linalg.lstsq(X[idx], y[idx], rcond=None)[0]
    th_b[b] = bb[2] + bb[3] + bb[4]
se_boot = th_b.std(ddof=1)
print("bootstrap SE (B=%d) =" % B, se_boot)
lo, hi = np.quantile(th_b, [0.025, 0.975])
print("percentile 95% CI =", lo, hi)
# BC
p0 = np.mean(th_b < theta)
from scipy.stats import norm
z0 = norm.ppf(np.clip(p0, 1e-6, 1 - 1e-6))
za = norm.ppf(0.025)
a1, a2 = norm.cdf(2 * z0 + za), norm.cdf(2 * z0 - za)
print("BC approx endpoints quantiles", a1, a2, "CI", np.quantile(th_b, [a1, a2]))


## 10.31 DDK cluster bootstrap（骨架，$B$ 可增大）

In [ ]:

ddk = pd.read_excel(ROOT / "DDK2011/DDK2011.xlsx")
for c in ddk.columns:
    ddk[c] = pd.to_numeric(ddk[c], errors="coerce")
ddk["ystd"] = (ddk.totalscore - ddk.totalscore.mean()) / ddk.totalscore.std()
d = ddk[["ystd", "tracking", "agetest", "girl", "etpteacher", "percentile", "schoolid"]].dropna()
schools = d.schoolid.unique()
y = d.ystd.values
X = np.column_stack([d.tracking, d.agetest, d.girl, d.etpteacher, d.percentile, np.ones(len(d))])
beta = np.linalg.lstsq(X, y, rcond=None)[0]
print("point beta", beta)

B = 200
rng = np.random.default_rng(1)
betas = np.zeros((B, X.shape[1]))
# map school -> rows
from collections import defaultdict
rows = defaultdict(list)
for i, g in enumerate(d.schoolid.values):
    rows[g].append(i)
sch = list(rows.keys())
for b in range(B):
    draw = rng.choice(sch, size=len(sch), replace=True)
    idx = np.concatenate([rows[g] for g in draw])
    betas[b] = np.linalg.lstsq(X[idx], y[idx], rcond=None)[0]
se = betas.std(axis=0, ddof=1)
print("cluster bootstrap SE", se)
print("BCa/percentile intervals can be formed columnwise from betas")
